# Stop and Search Analysis (February 2023 – January 2026)

Analysis of UK police stop and search incidents using open data. This notebook covers data loading, cleaning, and exploratory analysis across time, geography, gender, age, and ethnicity.

## 1. Setup & Data Loading

In [943]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix 
from jinja2 import Template

# widen display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

# Consistent plot styling
sns.set_theme(style='white')
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 13, 'axes.labelsize': 11})

# Load raw data
df_raw = pd.read_csv('all-stop-and-search.csv')

# Remove exact duplicate rows
df_clean = df_raw.drop_duplicates()

print(f'Raw rows:   {df_raw.shape[0]:,}')
print(f'After dedup: {df_clean.shape[0]:,}  ({df_raw.shape[0] - df_clean.shape[0]:,} duplicates removed)')


Raw rows:   43,738
After dedup: 43,081  (657 duplicates removed)


## 2. Data Inspection

In [944]:
# Schema and non-null counts
df_clean.info()


<class 'pandas.core.frame.DataFrame'>
Index: 43081 entries, 0 to 43737
Data columns (total 15 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Type                                      43081 non-null  object 
 1   Date                                      43081 non-null  object 
 2   Part of a policing operation              0 non-null      float64
 3   Policing operation                        0 non-null      float64
 4   Latitude                                  42795 non-null  float64
 5   Longitude                                 42795 non-null  float64
 6   Gender                                    40163 non-null  object 
 7   Age range                                 37875 non-null  object 
 8   Self-defined ethnicity                    43081 non-null  object 
 9   Officer-defined ethnicity                 42446 non-null  object 
 10  Legislation                            

In [945]:
df_clean.head()

,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,Legislation,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing
0,Person search,2023-02-01T00:23:00+00:00,NaN,NaN,5.154169300000e+01,-3.752000000000e-03,NaN,10-17,White - Any other White background,White,Police and Criminal Evidence Act 1984 (section 1),Anything to threaten or harm anyone,A no further action disposal,False,False
1,Person search,2023-02-01T01:35:00+00:00,NaN,NaN,5.139916800000e+01,-1.001130000000e-01,Female,18-24,White - English/Welsh/Scottish/Northern Irish/...,White,Police and Criminal Evidence Act 1984 (section 1),Anything to threaten or harm anyone,A no further action disposal,True,False
2,Person search,2023-02-01T09:53:00+00:00,NaN,NaN,5.153992800000e+01,8.028800000000e-02,Male,10-17,White - Any other White background,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,False,False
3,Person search,2023-02-01T11:00:00+00:00,NaN,NaN,5.320856200000e+01,-2.891456000000e+00,Male,25-34,White - Irish,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,True,False
4,Person search,2023-02-01T11:30:00+00:00,NaN,NaN,5.320856200000e+01,-2.891456000000e+00,Male,over 34,White - English/Welsh/Scottish/Northern Irish/...,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,True,False


In [946]:
df_clean.describe(include='all')

,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,Legislation,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing
count,43081,43081,0.000000000000e+00,0.000000000000e+00,4.279500000000e+04,4.279500000000e+04,40163,37875,43081,42446,42460,42692,43046,43048,43048
unique,3,39294,NaN,NaN,NaN,NaN,3,5,17,4,7,12,7,2,2
top,Person search,2024-10-04T16:00:00+00:00,NaN,NaN,NaN,NaN,Male,18-24,White - English/Welsh/Scottish/Northern Irish/...,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,False,False
freq,42752,7,NaN,NaN,NaN,NaN,36554,11495,17066,24106,28137,27905,36766,25762,42798
mean,NaN,NaN,NaN,NaN,5.213491932184e+01,-8.362106836546e-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,1.107547227619e+00,1.189161176243e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,5.014832400000e+01,-5.659012000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,5.149642000000e+01,-1.853315000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,5.153777900000e+01,-1.584410000000e-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,5.258232800000e+01,-8.649300000000e-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Data Cleaning

Steps:
1. Drop columns that are 100% null (`Part of a policing operation`, `Policing operation`).
2. Fill remaining categorical NaNs with `'Not recorded'`.
3. Parse `Date` into a proper datetime and extract time features.
4. Standardise `Age range` values (e.g. Excel-mangled `oct-17` → `10-17`).
5. Note: 286 rows have no latitude/longitude — these are kept to protect the inclusion of the remainiing data but excluded from geographic plots below.

In [947]:
# --- Data Cleaning Steps ---
# Convert Date to datetime
df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')

# Drop missing Latitude/Longitude
df = df_clean.dropna(subset=['Latitude', 'Longitude']).copy()

# Fix boolean column
df['Outcome linked to object of search'] = df['Outcome linked to object of search'].map({
    'True': True,
    'False': False,
    'Not recorded': None
})

# Replace 'Not recorded' values
df.replace(['Not recorded', 'not recorded'], pd.NA, inplace=True)

# Fix Month column
df['Month'] = df['Date'].dt.to_period('M').astype(str)

df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
Index: 42795 entries, 0 to 43737
Data columns (total 16 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   Type                                      42795 non-null  object             
 1   Date                                      42795 non-null  datetime64[ns, UTC]
 2   Part of a policing operation              0 non-null      float64            
 3   Policing operation                        0 non-null      float64            
 4   Latitude                                  42795 non-null  float64            
 5   Longitude                                 42795 non-null  float64            
 6   Gender                                    39893 non-null  object             
 7   Age range                                 37617 non-null  object             
 8   Self-defined ethnicity                    42795 non-null  obj

,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,Legislation,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing,Month
0,Person search,2023-02-01 00:23:00+00:00,NaN,NaN,5.154169300000e+01,-3.752000000000e-03,NaN,10-17,White - Any other White background,White,Police and Criminal Evidence Act 1984 (section 1),Anything to threaten or harm anyone,A no further action disposal,NaN,False,2023-02
1,Person search,2023-02-01 01:35:00+00:00,NaN,NaN,5.139916800000e+01,-1.001130000000e-01,Female,18-24,White - English/Welsh/Scottish/Northern Irish/...,White,Police and Criminal Evidence Act 1984 (section 1),Anything to threaten or harm anyone,A no further action disposal,NaN,False,2023-02
2,Person search,2023-02-01 09:53:00+00:00,NaN,NaN,5.153992800000e+01,8.028800000000e-02,Male,10-17,White - Any other White background,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,NaN,False,2023-02
3,Person search,2023-02-01 11:00:00+00:00,NaN,NaN,5.320856200000e+01,-2.891456000000e+00,Male,25-34,White - Irish,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,NaN,False,2023-02
4,Person search,2023-02-01 11:30:00+00:00,NaN,NaN,5.320856200000e+01,-2.891456000000e+00,Male,over 34,White - English/Welsh/Scottish/Northern Irish/...,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,A no further action disposal,NaN,False,2023-02


In [948]:
# 3a. Replace 'Not recorded' variants BEFORE dropping null columns
df_clean = df_clean.replace(
    ['Not recorded', 'not recorded', 'NOT RECORDED', 'Not Recorded'],
    pd.NA
)

# 3b. Drop always-null columns
cols_all_null = [c for c in df_clean.columns if df_clean[c].isna().all()]
print('Dropping fully-null columns:', cols_all_null)
df_clean = df_clean.drop(columns=cols_all_null)

# 3c. Fill categorical NaNs
cat_cols = df_clean.select_dtypes(include=['object', 'string']).columns
df_clean[cat_cols] = df_clean[cat_cols].fillna('not recorded')

# 3d. Parse Date and engineer time features
df_clean['Date'] = pd.to_datetime(df_clean['Date'], utc=True)
df_clean['Year']      = df_clean['Date'].dt.year
df_clean['Month']     = df_clean['Date'].dt.to_period('M').astype(str)
df_clean['Hour']      = df_clean['Date'].dt.hour
df_clean['DayOfWeek'] = pd.Categorical(
    df_clean['Date'].dt.day_name(),
    categories=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'],
    ordered=True
)

# 3e. Standardise Age range
age_map = {
    'oct-17':       '10-17',
    'under 10':     'under 10',
    '10-17':        '10-17',
    '18-24':        '18-24',
    '25-34':        '25-34',
    'over 34':      'over 34',
    'not recorded': 'not recorded'
}

df_clean['Age range'] = (
    df_clean['Age range']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(age_map)
)

# Working copy
df_analysis = df_clean.copy()

# Drop date column as we have extracted all relevant time features
df_analysis.drop(columns=['Date'], inplace=True)

# Verify remaining nulls
remaining = df_analysis.isna().sum()
print('\nRemaining NaN counts:')
print(remaining[remaining > 0])



Dropping fully-null columns: ['Part of a policing operation', 'Policing operation']

Remaining NaN counts:
Latitude     286
Longitude    286
dtype: int64


In [949]:
df_analysis.shape


(43081, 16)

In [950]:
df_analysis.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43081 entries, 0 to 43737
Data columns (total 16 columns):
 #   Column                                    Non-Null Count  Dtype   
---  ------                                    --------------  -----   
 0   Type                                      43081 non-null  object  
 1   Latitude                                  42795 non-null  float64 
 2   Longitude                                 42795 non-null  float64 
 3   Gender                                    43081 non-null  object  
 4   Age range                                 43081 non-null  object  
 5   Self-defined ethnicity                    43081 non-null  object  
 6   Officer-defined ethnicity                 43081 non-null  object  
 7   Legislation                               43081 non-null  object  
 8   Object of search                          43081 non-null  object  
 9   Outcome                                   43081 non-null  object  
 10  Outcome linked to object of

## 4. Temporal Patterns

### 4.1 By Month

In [951]:
month_counts = (
    df_analysis['Month']
    .astype('period[M]')
    .value_counts()
    .sort_index()
)

top3 = month_counts.nlargest(3).index

colors = []
for m in month_counts.index:
    if m == top3[0]:   colors.append('#2ca02c')    # busiest
    elif m == top3[1]: colors.append('#ff7f0e')    # 2nd
    elif m == top3[2]: colors.append('#d62728')    # 3rd
    else:              colors.append('steelblue')

fig, ax = plt.subplots(figsize=(16, 5))co
ax.bar(month_counts.index.astype(str), month_counts.values, color=colors)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Incidents')
ax.set_title('Stop and Search Incidents by Month (Feb 2023 – Jan 2026)')
ax.tick_params(axis='x', rotation=45)

from matplotlib.patches import Patch
legend = [
    Patch(color='#2ca02c', label='Busiest'),
    Patch(color='#ff7f0e', label='2nd'),
    Patch(color='#d62728', label='3rd'),
    Patch(color='steelblue', label='Other')
]
ax.legend(handles=legend, loc='upper right')

plt.tight_layout()
plt.show()


SyntaxError: invalid syntax (2253115992.py, line 17)

In [ ]:
# -----------------------------
# 2. MONTHLY SEASONAL ANALYSIS
# -----------------------------

# Extract month name + number for proper ordering
df_analysis['MonthName'] = pd.to_datetime(df_clean['Month']).dt.month_name()
df_analysis['MonthNumber'] = pd.to_datetime(df_clean['Month']).dt.month

# Count incidents per month across all years
month_counts = (
    df_analysis
    .groupby(['MonthNumber', 'MonthName'])
    .size()
    .sort_index()   # ensures Jan → Dec
)

# Flatten index to MonthName only (fixes colour issue)
month_counts.index = month_counts.index.get_level_values('MonthName')

# Identify top 3 busiest months
top3 = month_counts.nlargest(3).index

# Colour coding
colors = [
    '#2ca02c' if m == top3[0] else
    '#ff7f0e' if m == top3[1] else
    '#d62728' if m == top3[2] else
    'steelblue'
    for m in month_counts.index
]


# -----------------------------
# 3. PLOT WITH % LABELS
# -----------------------------

fig, ax = plt.subplots(figsize=(16, 5))

month_names = month_counts.index
values = month_counts.values
total = values.sum()

ax.bar(month_names, values, color=colors)

# Add % labels above each bar
for i, v in enumerate(values):
    pct = (v / total) * 100
    ax.text(
        i, v + (max(values) * 0.01),   # position slightly above bar
        f"{pct:.1f}%", 
        ha='center', va='bottom', fontsize=10
    )

ax.set_xlabel('Month')
ax.set_ylabel('Number of Incidents')
ax.set_title('Stop and Search Incidents by Month (All Years Combined)')
ax.tick_params(axis='x', rotation=45)

from matplotlib.patches import Patch
legend = [
    Patch(color='#2ca02c', label='Busiest'),
    Patch(color='#ff7f0e', label='2nd Busiest'),
    Patch(color='#d62728', label='3rd Busiest'),
    Patch(color='steelblue', label='Other')
]
ax.legend(handles=legend, loc='upper right')

plt.tight_layout()
plt.show()



In [ ]:
# Chi-square to test whether overall the counts are statistically significantly different by month (doesn't predict which months are significnatly higher/lower)

from scipy.stats import chisquare

# month_counts is your 12‑month total (already computed earlier)
observed = month_counts.values

chi2, p = chisquare(observed)

print("Chi-square statistic:", chi2)
print("p-value:", p)


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------
# 1. BUILD FULL P‑VALUE MATRIX
# -----------------------------

months = month_counts.index
values = month_counts.values
total = values.sum()

sig_matrix = pd.DataFrame(
    np.zeros((len(months), len(months))),
    index=months,
    columns=months
)

for i, m1 in enumerate(months):
    for j, m2 in enumerate(months):
        if i == j:
            sig_matrix.loc[m1, m2] = 1.0
        else:
            count = [month_counts[m1], month_counts[m2]]
            nobs = [total, total]
            stat, p = proportions_ztest(count, nobs)
            sig_matrix.loc[m1, m2] = p

# Show TRUE p-values in scientific notation
pd.set_option('display.float_format', lambda x: f'{x:.12e}')
print(sig_matrix)




In [ ]:
# Highlight which months are significantly different from each other at p < 0.05  import seaborn as sns
import matplotlib.pyplot as plt
sig_mask = sig_matrix < 0.05

plt.figure(figsize=(12, 10))

sns.heatmap(
    sig_mask,
    cmap=sns.color_palette(["#2ca02c", "#d62728"]),  # green = NS, red = sig
    cbar=False,
    linewidths=0.5,
    linecolor='black'
)

plt.title("Significant Month-to-Month Differences (p < 0.05)")
plt.show()


### 4.2 By Year

In [ ]:
# --- 1. Year counts and percentages ---
year_counts = (
    df_analysis['Year']
    .astype(int)
    .value_counts()
    .sort_index()
)

year_pct = (year_counts / year_counts.sum()) * 100

# --- 2. Convert to numpy arrays (forces numeric axis) ---
years = year_pct.index.to_numpy(dtype=int)
pct   = year_pct.to_numpy()

# --- 3. Identify top 3 busiest years ---
top3 = years[np.argsort(pct)[-3:]]  # last 3 = largest

# --- 4. Assign colours ---
def pick_color(y):
    if y == top3[-1]: return '#2ca02c'   # busiest
    if y == top3[-2]: return '#ff7f0e'   # 2nd
    if y == top3[-3]: return '#d62728'   # 3rd
    return 'steelblue'

colors = [pick_color(y) for y in years]

# --- 5. Plot ---
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(years, pct, color=colors)

ax.set_xlabel('Year')
ax.set_ylabel('Percentage of Incidents (%)')
ax.set_title('Stop and Search Incidents by Year (as % of total)')
ax.tick_params(axis='x', rotation=45)

# --- 6. Add % labels ---
for x, y in zip(years, pct):
    ax.text(x, y + 0.5, f"{y:.1f}%", ha='center', fontsize=10)

# --- 7. Legend ---
from matplotlib.patches import Patch
legend = [
    Patch(color='#2ca02c', label='Busiest'),
    Patch(color='#ff7f0e', label='2nd'),
    Patch(color='#d62728', label='3rd'),
    Patch(color='steelblue', label='Other')
]
ax.legend(handles=legend, loc='upper right')

plt.tight_layout()
plt.show()



### 4.3 By Day of Week

In [ ]:
day_counts = df_analysis['DayOfWeek'].value_counts().sort_index()
day_percent = day_counts / day_counts.sum() * 100
top_day = day_counts.idxmax()
colors = ['#2ca02c' if d == top_day else 'steelblue' for d in day_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(day_counts.index, day_percent.values, color=colors)
for bar, val in zip(bars, day_percent.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Day of Week')
ax.set_ylabel('% of Incidents')
ax.set_title('Stop and Search Incidents by Day of Week (Feb 2023 – Jan 2026)')
plt.tight_layout()
plt.show()


### 4.4 By Hour of Day

In [ ]:
# convert 24 hors to am/pm
def hour_to_label(h):
    if h == 0:
        return "12am"
    elif 1 <= h < 12:
        return f"{h}am"
    elif h == 12:
        return "12pm"
    else:
        return f"{h-12}pm"

# Add tidy label column
df_analysis['HourLabel'] = df_analysis['Hour'].apply(hour_to_label)

hour_counts = df_analysis['Hour'].value_counts().sort_index()
hour_percent = hour_counts / hour_counts.sum() * 100
top3_hours = hour_counts.nlargest(3).index
colors = ['#2ca02c' if h in top3_hours else 'steelblue' for h in hour_counts.index]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(hour_counts.index, hour_percent.values, color=colors)

for bar, val in zip(bars, hour_percent.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(hour_counts.index)
ax.set_xticklabels([hour_to_label(h) for h in hour_counts.index])

ax.set_xlabel('Hour of Day')
ax.set_ylabel('% of Incidents')
ax.set_title('Stop and Search Incidents by Hour of Day (Feb 2023 – Jan 2026)')

# Morning shading (0–11)
ax.axvspan(0, 12, color='lightgrey', alpha=0.2)

# Afternoon shading (12–24)
ax.axvspan(12, 24, color='lightyellow', alpha=0.2)

ax.text(6, ax.get_ylim()[1] * 0.95, "Morning", ha='center', fontsize=10)
ax.text(18, ax.get_ylim()[1] * 0.95, "Afternoon", ha='center', fontsize=10)


plt.tight_layout()
plt.show()




## 5. Geographic Distribution

In [ ]:
# Exclude rows without coordinates
df_geo = df_analysis.dropna(subset=['Latitude', 'Longitude'])
print(f'Plotting {len(df_geo):,} incidents with known coordinates ')
print(f'({len(df_analysis) - len(df_geo):,} excluded due to missing coordinates)')

fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=df_geo, x='Longitude', y='Latitude',
                alpha=0.2, s=5, color='steelblue', ax=ax)
ax.set_title('Geographic Distribution of Stop and Search Incidents')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()


## 6. Demographics

### 6.1 Gender

In [ ]:
gender_counts = df_analysis['Gender'].value_counts()
gender_percent = gender_counts / gender_counts.sum() * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(gender_counts.index, gender_percent.values, color='steelblue')
for bar, val in zip(bars, gender_percent.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11)
ax.set_xlabel('Gender')
ax.set_ylabel('% of Incidents')
ax.set_title('Stop and Search Incidents by Gender (Feb 2023 – Jan 2026)')
plt.tight_layout()
plt.show()


### 6.2 Age Group

In [ ]:
age_order = ['under 10', '10-17', '18-24', '25-34', 'over 34', 'not recorded']
age_counts = df_analysis['Age range'].value_counts().reindex(age_order, fill_value=0)
age_percent = age_counts / age_counts.sum() * 100
top_age = age_counts.idxmax()
colors = ['#2ca02c' if a == top_age else 'steelblue' for a in age_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(age_counts.index, age_percent.values, color=colors)
for bar, val in zip(bars, age_percent.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11)
ax.set_xlabel('Age Group')
ax.set_ylabel('% of Incidents')
ax.set_title('Stop and Search Incidents by Age Group (Feb 2023 – Jan 2026)')
plt.tight_layout()
plt.show()


### 6.3 Ethnicity

### 6.3.1 Officer defined ethnicity

In [ ]:
counts = df_analysis['Officer-defined ethnicity'].value_counts().sort_values(ascending=True)
pct = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(counts.index, pct.values, color='steelblue')
for bar, val in zip(bars, pct.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=10)
ax.set_xlabel('% of Incidents')
ax.set_ylabel('Ethnicity')
ax.set_title('Stop and Search by Officer-Defined Ethnicity (Feb 2023 – Jan 2026)')
plt.tight_layout()
plt.show()

### 6.3.2 Self-defined ethnicity

In [ ]:
# Self-defined ethnicity
counts = df_analysis['Self-defined ethnicity'].value_counts().sort_values(ascending=True)
pct = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(counts.index, pct.values, color='steelblue')
for bar, val in zip(bars, pct.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=10)
ax.set_xlabel('% of Incidents')
ax.set_ylabel('Ethnicity')
ax.set_title('Stop and Search by Self-defined Ethnicity (Feb 2023 – Jan 2026)')
plt.tight_layout()
plt.show()

## 7. Legislation


### 7.1 Legislation

In [ ]:
# Legislation
counts = df_analysis['Legislation'].value_counts().sort_values(ascending=True)
pct = counts / counts.sum() * 100   
fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(counts.index, pct.values, color='steelblue')
for bar, val in zip(bars, pct.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=10)
ax.set_xlabel('% of Incidents')
ax.set_ylabel('Legislation')
ax.set_title('Stop and Search by Legislation (Feb 2023 – Jan 2026)')
plt.tight_layout()  

### 7.2 Object of Search

In [ ]:
counts = df_analysis['Object of search'].value_counts().sort_values(ascending=True)
pct = counts / counts.sum() * 100   
fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(counts.index, pct.values, color='steelblue')
for bar, val in zip(bars, pct.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=10)
ax.set_xlabel('% of Incidents')
ax.set_ylabel('Objects of Search')
ax.set_title('Stop and Search by Legislation (Feb 2023 – Jan 2026)')
plt.tight_layout() 

### 7.3 Outcome

In [ ]:
counts = df_analysis['Outcome'].value_counts().sort_values(ascending=True)
pct = counts / counts.sum() * 100   
fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(counts.index, pct.values, color='steelblue')
for bar, val in zip(bars, pct.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=10)
ax.set_xlabel('% of Incidents')
ax.set_ylabel('Outcomes')
ax.set_title('Stop and Search by Legislation (Feb 2023 – Jan 2026)')
plt.tight_layout() 

### 7.4 Outcome linked to object of search - Boolean

In [ ]:
counts = df_analysis['Outcome linked to object of search'].astype(str).value_counts().sort_values(ascending=True)
pct = counts / counts.sum() * 100

# Rename directly without map
pct.index = ['Not recorded', 'Yes', 'No']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.barh(pct.index, pct.values, color='steelblue')
for bar, val in zip(bars, pct.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=11)
ax.set_xlabel('% of Incidents')
ax.set_ylabel('Outcome Linked to Object of Search')
ax.set_title('Was the Outcome Linked to the Object of Search? (Feb 2023 – Jan 2026)')
ax.set_facecolor('white')
ax.grid(False)
ax.set_xlim(0, 110)
plt.tight_layout()
plt.show()


### 7.5 Outcome linked to removal of more than just outer clothing?

In [ ]:
counts = df_analysis['Removal of more than just outer clothing'].astype(str).value_counts().sort_values(ascending=True)
pct = counts / counts.sum() * 100

# Rename directly without map
pct.index = ['Not recorded', 'Yes', 'No']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.barh(pct.index, pct.values, color='steelblue')
for bar, val in zip(bars, pct.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left', va='center', fontsize=11)
ax.set_xlabel('% of Incidents')
ax.set_ylabel('Removal of more than just outer clothing')
ax.set_title('Removal of More Than Just Outer Clothing? (Feb 2023 – Jan 2025)')
ax.set_facecolor('white')
ax.grid(False)
ax.set_xlim(0, 110)
plt.tight_layout()
plt.show()


## 8. Cross-tabulations and relationships between variables

Row-normalised percentages show the age distribution *within* each gender.

In [ ]:

age_order = ['under 10', '10-17', '18-24', '25-34', 'over 34', 'not recorded']

# Row‑normalised % table
ct = pd.crosstab(
    df_analysis['Gender'],
    df_analysis['Age range'],
    normalize='index'
) * 100

ct = ct.reindex(columns=age_order)

# Raw counts table (needed for totals)
gender_age_counts = pd.crosstab(
    df_analysis['Gender'],
    df_analysis['Age range']
)

# Insert total counts as first column
ct.insert(0, 'Total', gender_age_counts.sum(axis=0))


# --- Row-wise min/max highlighter ---
def highlight_row_minmax(s):
    # s is a row (axis=1)
    # Remove Total column from min/max logic
    row = s.drop(labels=["Total"], errors="ignore")

    is_max = row == row.max()
    is_min = row == row.min()

    styles = []
    for col in s.index:
        if col == "Total":
            styles.append("")  # no styling for Total
        elif is_max.get(col, False):
            styles.append("background-color: lightgreen; color: black")
        elif is_min.get(col, False):
            styles.append("background-color: #f5b5b5; color: black")
        else:
            styles.append("")
    return styles

styled_ct = ct.style.apply(highlight_row_minmax, axis=1)
styled_ct


## Row high and low %

In [ ]:
age_order = ['under 10', '10-17', '18-24', '25-34', 'over 34', 'not recorded']

# Row‑normalised % table
ct = pd.crosstab(
    df_analysis['Gender'],
    df_analysis['Age range'],
    normalize='index'
) * 100

ct = ct.reindex(columns=age_order)

# Raw counts table (needed for totals)
gender_age_counts = pd.crosstab(
    df_analysis['Gender'],
    df_analysis['Age range']
)

# Insert total counts as first column
ct.insert(0, 'Total', gender_age_counts.sum(axis=1))


# --- Column-wise min/max highlighter ---
def highlight_col_minmax(col):
    if col.name == "Total":   # skip Total column entirely
        return [""] * len(col)

    max_val = col.max()
    min_val = col.min()

    return [
        "background-color: lightgreen; color: black" if v == max_val else
        "background-color: #f5b5b5; color: black" if v == min_val else
        ""
        for v in col
    ]

# Apply column-wise
styled_ct = ct.style.apply(
    highlight_col_minmax,
    axis=0
)

styled_ct


# Apply to all columns except Total
cols_to_highlight = [c for c in ct.columns if c != "Total"]

styled_ct = ct.style.apply(
    highlight_col_minmax,
    axis=0,
    subset=cols_to_highlight
)

styled_ct

## pivot and look to column %

Does outcome vary by ethnicity? (e.g. are certain groups more likely to get "no further action"?)

In [ ]:
# Outcome by ethnicity (row-normalised %)
ct = pd.crosstab(
    df_analysis['Outcome'],
    df_analysis['Officer-defined ethnicity'],
    normalize='index'
) * 100

# Reorder ethnicity columns sensibly
ethnicity_order = ['White', 'Black', 'Asian', 'Other', 'Not recorded']
ct = ct.reindex(columns=[c for c in ethnicity_order if c in ct.columns])
ct.round(1)
print(ct.round(1))

def highlight_row_minmax(s):
    # s is a row (axis=1)
    is_max = s == s.max()
    is_min = s == s.min()

    return [
        "background-color: lightgreen; color: black" if is_max.iloc[i] else
        "background-color: #f5b5b5; color: black" if is_min.iloc[i] else
        ""
        for i in range(len(s))
    ]

styled_ct = ct.style.apply(highlight_row_minmax, axis=1)
styled_ct

# row %

In [ ]:
# Column‑normalised % table
ct_col = pd.crosstab(
    df_analysis['Outcome'],
    df_analysis['Officer-defined ethnicity'],
    normalize='columns'
) * 100

ct_col = ct_col.reindex(columns=[c for c in ethnicity_order if c in ct_col.columns]).round(1)

def highlight_col_minmax(s):
    is_max = s == s.max()
    is_min = s == s.min()
    return [
        "background-color: lightgreen; color: black" if is_max.iloc[i] else
        "background-color: #f5b5b5; color: black" if is_min.iloc[i] else
        ""
        for i in range(len(s))
    ]

styled_col = ct_col.style.apply(highlight_col_minmax, axis=0)
styled_col


Does object of search vary by age or gender?

In [ ]:
# Outcome by Gender and Age range (row-normalised %)
ct = pd.crosstab(
    df_analysis['Outcome'],
    [df_analysis['Gender'], df_analysis['Age range']],
    normalize='index'
) * 100

# Reorder columns
age_order = ['under 10', '10-17', '18-24', '25-34', 'over 34', 'not recorded']
gender_order = ['Male', 'Female', 'Not recorded', 'Other']

# Reindex with MultiIndex
new_cols = [(g, a) for g in gender_order for a in age_order 
            if (g, a) in ct.columns]
ct = ct.reindex(columns=new_cols)
ct.round(1)

Are certain legislation types used more on specific ethnic group types?

### Do different ages have differnt search profiles? IN reality I would exclude under 10 from analysis as base too low = 29. Remain for completeness

In [ ]:
df_analysis['Age range'].value_counts()

In [ ]:
age_order = [
    'under 10',
    '10-17',
    '18-24',
    '25-34',
    'over 34',
    'not recorded'
]
df_analysis['Age range'] = pd.Categorical(
    df_analysis['Age range'],
    categories=age_order,
    ordered=True
)
age_object_counts = pd.crosstab(
    df_analysis['Age range'],
    df_analysis['Object of search']
).loc[age_order]

print("Age × Object of Search — Counts (Ordered)")
display(age_object_counts)



In [ ]:
age_object_pct_rows = pd.crosstab(
    df_analysis['Age range'],
    df_analysis['Object of search'],
    normalize='index'
).round(3).loc[age_order]

# Add total count column on the left
age_object_pct_rows.insert(0, 'Total', age_object_counts.sum(axis=1))

age_object_pct_rows



In [ ]:
# Highlights the top and bottom % in each column
def highlight_min_max(s):
    is_max = s == s.max()
    is_min = s == s.min()

    return [
        "background-color: #b6e3b6; color: black" if is_max.iloc[i] else
        "background-color: #f5b5b5; color: black" if is_min.iloc[i] else
        ""
        for i in range(len(s))
    ]



# Apply styling (skip the 'Total' column)
cols_to_highlight = [
    c for c in age_object_pct_rows.columns
    if c not in ["Total", "Crossbows"]
]

styled_age_object_rows = (
    age_object_pct_rows.style
        .apply(
            highlight_min_max,
            subset=cols_to_highlight,
            axis=0
        )
        .format("{:.3f}")
)

styled_age_object_rows




### Build the equivalent gender table

In [ ]:

# get the gender counts 
gender_order = ["Male", "Female", "not recorded", "Other"]

gender_object_counts = pd.crosstab(
    df_analysis['Gender'],
    df_analysis['Object of search']
).loc[gender_order]

gender_object_counts





In [ ]:
# Row-normalised percentage table
gender_object_pct_rows = pd.crosstab(
    df_analysis['Gender'],
    df_analysis['Object of search'],
    normalize='index'
).round(3).loc[gender_order]

# Add total count column on the left
gender_object_pct_rows.insert(0, 'Total', gender_object_counts.sum(axis=1))

gender_object_pct_rows



In [ ]:
# Column-normalised percentage table

# Highlights the top and bottom % in each column
def highlight_min_max(s):
    is_max = s == s.max()
    is_min = s == s.min()

    return [
        "background-color: #b6e3b6; color: black" if is_max.iloc[i] else
        "background-color: #f5b5b5; color: black" if is_min.iloc[i] else
        ""
        for i in range(len(s))
    ]


# Columns to highlight (skip Total + Crossbows)
cols_to_highlight_gender = [
    c for c in gender_object_pct_rows.columns
    if c not in ["Total", "Crossbows"]
]

# Apply styling
styled_gender_object_rows = (
    gender_object_pct_rows.style
        .apply(
            highlight_min_max,
            subset=cols_to_highlight_gender,
            axis=0
        )
        .format("{:.3f}")
)

styled_gender_object_rows


---
## 9. Statistical Testing

Chi-square tests, effect sizes (Cramér's V), and relative risk analysis to assess whether observed patterns are statistically significant.


Chi Square: To check if relationships eg ethnicity vs outomce are statistically significant?

## Chi Square: Observed Counts vs expected counts during day - Hypothesis -(one-tailed/directional)

## Null hypothesis : Incidents are equally likely in every hour
## Alternative hypothesis: Incidents are more likely to occur during the early to late evening than would be expected by chance

In [ ]:
from scipy.stats import chisquare

# Define evening window
evening_hours = list(range(15, 22))  # 15–21 inclusive

# Observed counts
observed_in_window = hour_counts.loc[evening_hours].sum()
observed_outside = hour_counts.sum() - observed_in_window

observed = [observed_in_window, observed_outside]

# Expected counts (uniform distribution)
expected_in_window = hour_counts.sum() * (len(evening_hours) / 24)
expected_outside = hour_counts.sum() - expected_in_window

expected = [expected_in_window, expected_outside]

chi2, p_two_tailed = chisquare(f_obs=observed, f_exp=expected)

# Convert to one-tailed p-value
p_one_tailed = p_two_tailed / 2

print("Observed:", observed)
print("Expected:", expected)
print("Chi-square:", chi2)
print("Two-tailed p:", p_two_tailed)
print("One-tailed p:", p_one_tailed)

# Results: a chi-square of 10 is considered strong - this is 15413, so very strong evidence of non-uniform distribution across hours. 
# The one-tailed p-value is effectively zero, confirming that the evening window has a significantly higher proportion of incidents 
# than expected under uniform distribution.
# observed incidents = 24,278; expected: 12,565

### Effect size: Cramers V - how strong the chi-square effect is

Evening (3pm-9pm) vs Non Evening (0-2pm, 10pm-11pm)

Result: large/very large impact - evening represents 7 hours (29% of the day) but 56% of the incidents
Conclusion: Afternoon/evening spike is real, large, and meaningful

In [ ]:
chi2 = 15413.611339710085
n = hour_counts.sum()

cramers_v = np.sqrt(chi2 / n)
print("Cramér's V:", cramers_v)


# Lets split the day up into e.g. 3 hour segments to support policing iniatives

In [ ]:
# 1. Define 3-hour bins and labels
bins = list(range(0, 25, 3))  # [0, 3, 6, ..., 24]
labels = [
    "00–03", "03–06", "06–09", "09–12",
    "12–15", "15–18", "18–21", "21–24"
]

# 2. Create a binned column from your Hour variable
df_analysis['HourBin'] = pd.cut(
    df_analysis['Hour'],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

# 3. Get counts per 3-hour bin
bin_counts = df_analysis['HourBin'].value_counts().sort_index()

print(bin_counts)

# 4. Chi-square on 3-hour bins
from scipy.stats import chisquare

observed = bin_counts.values
expected = [bin_counts.sum() / len(bin_counts)] * len(bin_counts)

chi2, p = chisquare(observed, expected)

print("Chi-square:", chi2)
print("p-value:", p)
   


In [ ]:
# 4. Chi-square on 3-hour bins
from scipy.stats import chisquare

observed = bin_counts.values
expected = [bin_counts.sum() / len(bin_counts)] * len(bin_counts)

chi2, p = chisquare(observed, expected)

print("Chi-square:", chi2)
print("p-value:", p)


# Results: this chi-square is 25748, so very strong evidence of non-uniform distribution across hours. 
 

### Effect size: Cramers V - how strong the chi-square effect is

In [ ]:
chi2 = 25748.3224391263
n = bin_counts.sum()

cramers_v = np.sqrt(chi2 / n)
print("Cramér's V:", cramers_v)


# Undestanding what was the reason for the search by the 3 hour bins

In [ ]:
# counts by 3 hour bins
df_analysis.groupby(['HourBin', 'Object of search']).size().unstack(fill_value=0)


In [ ]:
# % by 3 hour bins
pct_table = (
    df_analysis
    .groupby(['HourBin', 'Object of search'])
    .size()
    .groupby(level=0)
    .apply(lambda x: (x / x.sum()) * 100)
    .unstack(fill_value=0)
    .round(2)
)

pct_table.index.name = None  # remove duplicate HourBin label
pct_table



In [ ]:

overall_pct = (
    df_analysis['Object of search']
    .value_counts(normalize=True) * 100
).round(2)

overall_pct


# Lets flip the table and % by columns/object of search!

### Relative risk of controlled drugs - to identify whether 3-6pm is truly a 'special' time

In [ ]:
table = (
    df_analysis
    .groupby(['HourBin', 'Object of search'])
    .size()
    .unstack(fill_value=0)
)
cd = table['Controlled drugs']
cd_pct = (cd / cd.sum()) * 100
cd_1518 = cd.loc['15–18']
cd_other = cd.drop('15–18').sum()

total_1518 = table.sum(axis=1).loc['15–18']
total_other = table.sum(axis=1).drop('15–18').sum()

p_1518 = cd_1518 / total_1518
p_other = cd_other / total_other

RR = p_1518 / p_other
RR

#Conclusion: controlled drugs is not the reason for the 3-6pm peak. Although 31.42% of all drugs searches occur in the 3-6pm window, this is because it has the highest volume of stops overall.
# The relative risk of a drugs search in the 3-6pm window is actually slightly lower than in other hours (RR=0.95), so drugs searches are not disproportionately driving the evening peak. 

# Repeat Relative Risk for all objects

In [ ]:
# Focus on 3pm-6pm 
# # Compute RR for all objects, sort and put in a table
# Total stops in 15–18 and all other times
total_1518 = table.sum(axis=1).loc['15–18']
total_other = table.sum(axis=1).drop('15–18').sum()

rows = []

for obj in table.columns:
    obj_counts = table[obj]
    
    obj_1518 = obj_counts.loc['15–18']
    obj_other = obj_counts.drop('15–18').sum()
    
    p_1518 = obj_1518 / total_1518
    p_other = obj_other / total_other
    
    RR = p_1518 / p_other
    
    rows.append([obj, RR, obj_counts.sum()])

RR_df = (
    pd.DataFrame(rows, columns=['Object of search', 'Relative Risk', 'Total Count'])
    .sort_values('Relative Risk', ascending=False)
    .reset_index(drop=True)
    .round({'Relative Risk': 3})
)

RR_df

# Conclusion: For the 3-6pm window, Crossbows can be ignored due to extremely low counts.
# Offensive weapons have the highest meaningful relative risk, being 32.7% more likely to occur between 15:00 and 18:00 compared to all other times.
# This makes offensive weapons searches the strongest contributor to the 3–6pm peak.

# Now lets repeat for all 3-hour time slots and select top risk for each slot

In [ ]:
RR_rows = []

for bin_name in table.index:
    total_bin = table.sum(axis=1).loc[bin_name]
    total_other = table.sum(axis=1).drop(bin_name).sum()
    
    for obj in table.columns:
        obj_counts = table[obj]
        
        obj_bin = obj_counts.loc[bin_name]
        obj_other = obj_counts.drop(bin_name).sum()
        
        p_bin = obj_bin / total_bin
        p_other = obj_other / total_other
        
        RR = p_bin / p_other
        
        RR_rows.append([bin_name, obj, RR, obj_counts.sum()])

RR_full = (
    pd.DataFrame(RR_rows, columns=['HourBin', 'Object of search', 'Relative Risk', 'Total Count'])
    .sort_values(['HourBin', 'Relative Risk'], ascending=[True, False])
    .reset_index(drop=True)
)

# Extract the highest meaningful RR per bin

meaningful = RR_full[RR_full['Total Count'] >= 100]

top_per_bin = (
    meaningful
    .sort_values(['HourBin', 'Relative Risk'], ascending=[True, False])
    .groupby('HourBin')
    .head(1)
    .reset_index(drop=True)
)

top_per_bin

# Conclusion: The following shows the relative risk (how likely something is in one group versus another / over representation) by time of day with an associated count. 
# Relative risk shows the true time-of-day patterns
# Analysis of relative risk across the day reveals that different types of searches cluster strongly at specific time periods, 
# reflecting distinct behavioural and operational patterns on the rail network. Each 3‑hour window has a characteristic “risk signature”, 
# driven by the interaction of passenger flows, policing activity, and the nature of offences encountered.

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

# Data with friendly time labels
data = {
    'HourBin': [
        'midnight–3am',
        '3–6am',
        '6–9am',
        '9am–12pm',
        '12–3pm',
        '3–6pm',
        '6–9pm',
        '9pm–midnight'
    ],
    'RR': [3.334993, 8.497418, 2.815471, 2.252132, 1.086822, 1.326590, 1.303358, 1.647780],
    'RR_Group': [
        'Criminal damage items',
        'Criminal damage items',
        'Duty-unpaid goods',
        'Duty-unpaid goods',
        'Controlled drugs',
        'Offensive weapons',
        'Firearms',
        'Firearms'
    ]
}

df = pd.DataFrame(data)

plt.figure(figsize=(14,6))
bars = plt.bar(df['HourBin'], df['RR'], color='steelblue')

# Add RR values *inside* the bars
for bar, rr in zip(bars, df['RR']):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height/2,
        f"{rr:.2f}",
        ha='center',
        va='center',
        fontsize=10,
        color='white',
        fontweight='bold'
    )

# Add RR group labels *above* the bars
for bar, group in zip(bars, df['RR_Group']):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.15,
        group,
        ha='center',
        va='bottom',
        fontsize=9,
        rotation=0
    )

# Time labels below bars (horizontal)
plt.xticks(
    ticks=range(len(df)),
    labels=df['HourBin'],
    rotation=0,
    fontsize=10
)

plt.xlabel('Time of Day')
plt.ylabel('Relative Risk')
plt.title('Highest Meaningful Relative Risk by Time of Day')

plt.tight_layout()
plt.show()



---
## 10. Predictive Modelling

| Section | Content |
|---|---|
| **A** | Imports |
| **B** | Data Preparation |
| **C** | Baseline Model (DummyClassifier) |
| **D** | Binary Logistic Regression |
| **E** | Multinomial Logistic Regression |
| **F** | SHAP Values |


---
## A. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import shap
import warnings
warnings.filterwarnings('ignore')

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay, average_precision_score
)

sns.set_theme(style='white')
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 13, 'axes.labelsize': 11})
print('Imports OK')

---
## B. Data Preparation

All preparation before any model is fitted:
- **B1** Missing value audit & imputation
- **B2** Feature engineering & encoding
- **B3** VIF multicollinearity check
- **B4** Train / test split & class imbalance audit

In [ ]:
# ── B1. Missing value audit ───────────────────────────────────────────────────
df_prep = df_clean.copy()

FEATURES = ['Gender', 'Age range', 'Officer-defined ethnicity', 'Object of search', 'Date']
TARGETS  = ['Outcome linked to object of search', 'Outcome']

audit = df_prep[FEATURES + TARGETS].isnull().sum().rename('Missing').to_frame()
audit['% Missing'] = (audit['Missing'] / len(df_prep) * 100).round(2)
print(f'Total rows: {len(df_prep):,}\n')
print(audit.to_string())

In [ ]:
# ── B1 (cont). Imputation ─────────────────────────────────────────────────────
# Fill with explicit 'Unknown' rather than mode — missingness may carry signal.
for col in ['Gender', 'Age range', 'Officer-defined ethnicity', 'Object of search']:
    n = df_prep[col].isnull().sum()
    if n > 0:
        df_prep[col] = df_prep[col].fillna('Unknown')
        print(f'  {col}: {n:,} nulls → "Unknown"')

before = len(df_prep)
df_prep = df_prep.dropna(subset=TARGETS)
print(f'\nRows dropped (null targets only): {before - len(df_prep):,}')
print(f'Rows remaining: {len(df_prep):,}')

In [ ]:
# ── B2. Feature engineering & encoding ───────────────────────────────────────
df_prep['Date'] = pd.to_datetime(df_prep['Date'], utc=True)
df_prep['Hour'] = df_prep['Date'].dt.hour

# Binary target: True/False → 1/0
df_prep['outcome_binary'] = (
    df_prep['Outcome linked to object of search']
    .map({'True': 1, 'False': 0, True: 1, False: 0})
    .fillna(0)
    .astype(int)
)

# Multi-class target: collapse rare categories (<1%) into 'Other'
freq = df_prep['Outcome'].value_counts(normalize=True)
rare = freq[freq < 0.01].index
df_prep['outcome_multi'] = df_prep['Outcome'].apply(
    lambda x: 'Other' if x in rare else x
)
print('Outcome classes (multi-class target):')
print(df_prep['outcome_multi'].value_counts())

# One-hot encode — drop_first avoids dummy variable trap
X_raw = pd.get_dummies(
    df_prep[['Gender', 'Age range', 'Officer-defined ethnicity', 'Object of search', 'Hour']],
    columns=['Gender', 'Age range', 'Officer-defined ethnicity', 'Object of search'],
    drop_first=True
).astype(float)

print(f'\nFeature matrix: {X_raw.shape[0]:,} rows × {X_raw.shape[1]} features')

In [ ]:
# ── B3. VIF multicollinearity check ──────────────────────────────────────────
# Computed on a 5k sample for speed. VIF > 10 = problematic.
X_vif  = sm.add_constant(X_raw.sample(5000, random_state=42))

# Calculate VIF for each feature
vif_df = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF':     [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).set_index('Feature').drop('const')

# Flag features with VIF > 10 i.e. identify problematic features
flagged = vif_df[vif_df['VIF'] > 10]
print(f'Features with VIF > 10: {len(flagged)}')
print(flagged.sort_values('VIF', ascending=False).round(2).to_string() if len(flagged) else 'None ✓')

# Print top 20 features by VIF for inspection - green is acceptable, red is severe (>10)
fig, ax = plt.subplots(figsize=(10, 6))
top_vif = vif_df.sort_values('VIF', ascending=False).head(20)
colors  = ['#e05c5c' if v > 10 else '#5c8ee0' if v > 5 else '#a8c8a8' for v in top_vif['VIF']]
ax.barh(top_vif.index[::-1], top_vif['VIF'][::-1], color=colors[::-1])

# Draws refernce lines in the chart for VIF thresholds
ax.axvline(10, color='red',    linestyle='--', linewidth=1, label='VIF = 10 (severe)')
ax.axvline(5,  color='orange', linestyle='--', linewidth=1, label='VIF = 5 (moderate)')
ax.set_xlabel('Variance Inflation Factor')
ax.set_title('VIF — Top 20 Features', fontsize=12)
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

# No features have VIF > 10, so we can proceed without dropping any for multicollinearity.
# All features have VIF < 5, indicating low multicollinearity. 

In [ ]:
# Creates train/test splits for both your binary and multi‑class targets

# Uses stratification to preserve class proportions

# Sets up 5‑fold cross‑validation

# Audits class imbalance for both targets

# Visualises the imbalance

# Prints the binary imbalance ratio (e.g., 3.2:1)

# It’s essentially a data readiness check before modelling.

# ── B4. Train / test split & class imbalance audit ───────────────────────────
y_bin   = df_prep['outcome_binary']
y_multi = df_prep['outcome_multi']

X_tr, X_te, y_tr_bin, y_te_bin = train_test_split(
    X_raw, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)
_, _, y_tr_multi, y_te_multi = train_test_split(
    X_raw, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train: {len(X_tr):,}  |  Test: {len(X_te):,}')

# Class imbalance chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bin_counts = y_bin.value_counts().rename({0: 'Not Linked', 1: 'Linked'})
axes[0].bar(bin_counts.index, bin_counts.values, color=['#5c8ee0','#e05c5c'], edgecolor='white')
for i, v in enumerate(bin_counts.values):
    axes[0].text(i, v + 100, f'{v:,}\n({v/len(y_bin)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_title('Binary target')
axes[0].set_ylabel('Count')
sns.despine(ax=axes[0])

mc = y_multi.value_counts()
axes[1].barh(mc.index[::-1], mc.values[::-1], color='steelblue', edgecolor='white')
axes[1].set_title('Multi-class target')
axes[1].set_xlabel('Count')
sns.despine(ax=axes[1])

plt.suptitle('Class distribution  —  class_weight="balanced" applied to all models',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()
print(f'Binary imbalance ratio: {bin_counts.max()/bin_counts.min():.1f}:1')

# Chart shows mild imbalance between the Not linked and Linked.
# Implication for modelling
# No need for oversampling/undersampling. Balanced class weights are sufficient.

# Performance metrics (confusion matrix) should include precision, recall, F1, not just accuracy.

---
## C. Baseline Model

A `DummyClassifier` sets the performance floor before any real model is fitted.
Every subsequent model must beat this — if it doesn't, it has learned nothing useful.

In [ ]:
# ── C. Dummy classifier benchmark ────────────────────────────────────────────
dummy_rows = []
for strategy in ['most_frequent', 'stratified', 'uniform']:
    d   = DummyClassifier(strategy=strategy, random_state=42).fit(X_tr, y_tr_bin)
    yp  = d.predict(X_te)
    ypr = d.predict_proba(X_te)[:, 1]
    dummy_rows.append({
        'Model':    f'Dummy ({strategy})',
        'Accuracy': round((yp == y_te_bin).mean(), 4),
        'ROC-AUC':  round(roc_auc_score(y_te_bin, ypr), 4),
        'Avg Prec': round(average_precision_score(y_te_bin, ypr), 4),
    })

dummy_df = pd.DataFrame(dummy_rows).set_index('Model')
print('=== Dummy baseline performance ===')
print(dummy_df.to_string())

DUMMY_AUC = dummy_df['ROC-AUC'].max()
print(f'\nROC-AUC ceiling to beat: {DUMMY_AUC:.4f}')

# ROC-AUC tells how well the model can separate the two calsses: Linked and Not Linked (0.5-no skill, 0.6-0.7: weak byt learning something, 0.7-0.8: decent discrimination; 0.8-0.9: strong 
# model etc Here 1.0-perfect). If a model cannot beat the dummy AUC of 0.5, it’s not learning any meaningful patterns and is no better than random guessing.
# 3 models:
# most frequent - always predicts the majority class - average precision of 0.4013 - no learning, just predicting the most common class
# stratified classes randomly but in proportion to their frequency - average precision of 0.4001 - no real learning, just random guessing based on class distribution
# uniform predicts the classes uniformly at random - reflects class prevalence

# If the model cannot beat the ROC-AUC is 0.5, it’s not learning any meaningful patterns and is no better than random guessing.
# Summary: any real model must achieve ROC-AUC>0.5 to be better than chance at identifying the positive class (Linked). The dummy benchmarks set a baseline to surpass.

---
## D. Binary Logistic Regression

**Target:** `Outcome linked to object of search` (1 = linked, 0 = not linked)

The scaler lives inside a Pipeline so it is re-fit on each CV fold — no data leakage.
We tune first, then evaluate the tuned model on the held-out test set.

In [ ]:
# ── D1. Tune via GridSearchCV (Pipeline = no leakage) ────────────────────────
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
])

lr_grid = GridSearchCV(
    lr_pipe,
    param_grid={
        'lr__C':       [0.001, 0.01, 0.1, 1, 10],
        'lr__penalty': ['l1', 'l2'],
        'lr__solver':  ['liblinear'],
    },
    cv=CV, scoring='roc_auc', n_jobs=-1
)
lr_grid.fit(X_tr, y_tr_bin)
lr_tuned = lr_grid.best_estimator_

print(f'Best params:     {lr_grid.best_params_}')
print(f'Best CV ROC-AUC: {lr_grid.best_score_:.4f}')

# Regularisation sensitivity
cv_res = pd.DataFrame(lr_grid.cv_results_)
fig, ax = plt.subplots(figsize=(9, 4))
for pen, grp in cv_res.groupby('param_lr__penalty'):
    grp = grp.sort_values('param_lr__C')
    ax.semilogx(grp['param_lr__C'], grp['mean_test_score'],
                marker='o', label=f'penalty={pen}', linewidth=1.8)
    ax.fill_between(grp['param_lr__C'],
                    grp['mean_test_score'] - grp['std_test_score'],
                    grp['mean_test_score'] + grp['std_test_score'], alpha=0.12)
ax.axvline(lr_grid.best_params_['lr__C'], color='red', linestyle='--', linewidth=1,
           label=f'Best C={lr_grid.best_params_["lr__C"]}')
ax.set_xlabel('C  (lower = stronger regularisation)')
ax.set_ylabel('CV ROC-AUC')
ax.set_title('Binary LR — Regularisation sensitivity', fontsize=12)
ax.legend(fontsize=9)
sns.despine()
plt.tight_layout()
plt.show()

# chart shows: logistic regression model is learning meaningful signal (ROC‑AUC ≈ 0.713 is well above the dummy baseline of 0.50).

# L2 regularisation is the right choice for this dataset.

# C = 1 is a sensible, interpretable, and stable hyperparameter.

# The model is robust and not overly dependent on fine‑tuning.
# The logistic regression model works best with L2 regularisation and a C value of 1, giving a solid ROC‑AUC of about 0.713 — 
# clearly better than chance and very stable across settings.

In [ ]:
# ── D2. Evaluate tuned model on held-out test set ────────────────────────────
y_pred_lr = lr_tuned.predict(X_te)
y_prob_lr = lr_tuned.predict_proba(X_te)[:, 1]
auc_lr    = roc_auc_score(y_te_bin, y_prob_lr)

assert auc_lr > DUMMY_AUC, f'LR ({auc_lr:.4f}) does not beat dummy ({DUMMY_AUC:.4f})'

print('=== Classification Report — Tuned Binary LR ===')
print(classification_report(y_te_bin, y_pred_lr, target_names=['Not Linked', 'Linked']))
print(f'ROC-AUC:  {auc_lr:.4f}  (dummy ceiling: {DUMMY_AUC:.4f})')
print(f'Avg Prec: {average_precision_score(y_te_bin, y_prob_lr):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_te_bin, y_pred_lr), display_labels=['Not Linked', 'Linked']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Binary LR')

RocCurveDisplay.from_predictions(y_te_bin, y_prob_lr, ax=axes[1], color='steelblue')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Dummy')
axes[1].set_title('ROC Curve — Binary LR')
axes[1].legend()
plt.tight_layout()
plt.show()

# Summary: Model is not great at identifying Not Linked cases (recall 0.55) but is very good at identifying Linked cases (recall 0.85).
# The tuned logistic regression model is meaningfully better than chance (ROC‑AUC ≈ 0.705), and it’s especially good at identifying Linked cases.
#   However, it misses a lot of Not Linked cases, which is why accuracy sits around 0.64.
# model has correct recal for Linked cases but lower recall for Not Linked cases, which is common in imbalanced datasets. 
# The ROC‑AUC of about 0.705 confirms it’s learning real patterns, not just guessing.

In [ ]:
# ── D3. Odds ratios (statsmodels) ─────────────────────────────────────────────
# Re-scale training data using the best pipeline's scaler for sm.Logit.
scaler_d  = lr_tuned.named_steps['scaler']
X_tr_sc   = scaler_d.transform(X_tr)

# Inverse-frequency weights to replicate class_weight='balanced'
cc   = y_tr_bin.value_counts()
wmap = {c: len(y_tr_bin) / (2 * n) for c, n in cc.items()}
sw   = y_tr_bin.map(wmap).values

# 1. Build X with correct index
X_sm = pd.DataFrame(X_tr_sc, columns=X_raw.columns, index=y_tr_bin.index)
X_sm = sm.add_constant(X_sm)

# 2. Reset y to match
y = y_tr_bin.astype(int).reset_index(drop=True)
X_sm = X_sm.reset_index(drop=True)

# 3. Fit model
logit_fit = sm.Logit(y, X_sm).fit(disp=False, freq_weights=sw)


# ── FIXED CONFIDENCE INTERVALS ───────────────────────────────────────────────
ci = logit_fit.conf_int()

# Force into a DataFrame with 2 named columns
ci = pd.DataFrame(ci, columns=['CI Lower', 'CI Upper'])

# Then align the index to the parameter names
ci.index = logit_fit.params.index

# Finally, move to odds-ratio scale
ci = np.exp(ci)


# Build coefficient table (drop intercept)
coef_df = pd.DataFrame({
    'Odds Ratio': np.exp(logit_fit.params),
    'p-value':    logit_fit.pvalues,
    'CI Lower':   ci['CI Lower'],
    'CI Upper':   ci['CI Upper'],
}).iloc[1:]  # drop intercept

coef_df = coef_df.sort_values('Odds Ratio', ascending=False)

# Significant predictors
sig = coef_df[coef_df['p-value'] < 0.05]

# Show ALL significant predictors
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(sig.round(3).to_string())
print(f"Total significant predictors: {len(sig)}")

print(f'\nPseudo R² (McFadden): {logit_fit.prsquared:.4f}')

# ── Forest plot ──────────────────────────────────────────────────────────────
plot_df = sig.copy()
plot_df.index = (plot_df.index
    .str.replace('Object of search_', '', regex=False)
    .str.replace('Age range_', 'Age: ', regex=False)
    .str.replace('Officer-defined ethnicity_', '', regex=False)
    .str.replace('Gender_', '', regex=False))

fig, ax = plt.subplots(figsize=(10, 7))
ypos = range(len(plot_df))
ax.barh(list(ypos), plot_df['Odds Ratio'],
        color=['#e05c5c' if v > 1 else '#5c8ee0' for v in plot_df['Odds Ratio']], alpha=0.8)
ax.errorbar(plot_df['Odds Ratio'], list(ypos),
            xerr=[(plot_df['Odds Ratio'] - plot_df['CI Lower']).abs(),
                  (plot_df['CI Upper']   - plot_df['Odds Ratio']).abs()],
            fmt='none', color='black', capsize=3, linewidth=1)
ax.set_yticks(list(ypos))
ax.set_yticklabels(plot_df.index, fontsize=9)
ax.axvline(1, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Odds Ratio (95% CI) — standardised features')
ax.set_title('Significant Predictors — Binary LR (p < 0.05)', fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

# Summary: After running a logistic regression it has identified 17 features that are statistically significant (p < 0.05).
# These are the features that genuinely help the model decide whether a case is Linked or Not Linked.
# Odds ratio above 1 increases the liklihood of being Linked, an odds ration below 1 decreases the liklihood of being Linked

---
## E. Multinomial Logistic Regression

**Target:** `outcome_multi` — full outcome category (rare classes collapsed to 'Other')

Same structure as Section D: tune → evaluate → interpret.

In [ ]:
# ── E1. Tune via GridSearchCV ─────────────────────────────────────────────────
mlr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(multi_class='multinomial', solver='lbfgs',
                                  max_iter=1000, class_weight='balanced', random_state=42)),
])

mlr_grid = GridSearchCV(
    mlr_pipe,
    param_grid={'lr__C': [0.01, 0.1, 1, 10]},
    cv=CV, scoring='f1_macro', n_jobs=-1
)
mlr_grid.fit(X_tr, y_tr_multi)
mlr_tuned = mlr_grid.best_estimator_

print(f'Best params:      {mlr_grid.best_params_}')
print(f'Best CV F1-macro: {mlr_grid.best_score_:.4f}')

In [ ]:
# ── E2. Evaluate on held-out test set ─────────────────────────────────────────
y_pred_multi = mlr_tuned.predict(X_te)

print('=== Classification Report — Tuned Multinomial LR ===')
print(classification_report(y_te_multi, y_pred_multi, zero_division=0))

# Normalised confusion matrix
labels = sorted(y_multi.unique())
cm_m   = confusion_matrix(y_te_multi, y_pred_multi, labels=labels, normalize='true')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pd.DataFrame(cm_m, index=labels, columns=labels),
            annot=True, fmt='.2f', cmap='Blues', linewidths=0.5, ax=ax)
ax.set_title('Normalised Confusion Matrix — Tuned Multinomial LR', fontsize=12)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# Summary: The multinomial logistic regression model completely fails to predict disposal outcomes because the features contain almost no usable signal, 
# the classes are highly imbalanced, and the decision process is too complex and non‑linear for a simple linear classifier.

In [ ]:
# ── E3. Coefficient heatmap (feature × outcome class) ─────────────────────────
lr_step   = mlr_tuned.named_steps['lr']
coef_mat  = pd.DataFrame(lr_step.coef_, index=lr_step.classes_, columns=X_raw.columns)
top_feats = coef_mat.abs().max(axis=0).nlargest(15).index
coef_top  = coef_mat[top_feats].copy()
coef_top.columns = (coef_top.columns
    .str.replace('Object of search_', '', regex=False)
    .str.replace('Age range_', 'Age: ', regex=False)
    .str.replace('Officer-defined ethnicity_', '', regex=False)
    .str.replace('Gender_', '', regex=False))

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(coef_top, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', linewidths=0.4, ax=ax)
ax.set_title('Multinomial LR — Coefficients by Outcome Class (top 15 features)', fontsize=12)
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

# Heatmap confimrs the model cannot separate the classes

---
## F. SHAP Values

SHAP explains individual predictions from the binary LR — showing not just which features
matter globally, but the direction and magnitude of each feature's contribution to each
specific prediction.

We use `LinearExplainer` (appropriate for logistic regression) on a 1,000-row test sample.

In [ ]:
# ── F1. Compute SHAP values ───────────────────────────────────────────────────
# Extract the fitted scaler from the pipeline and transform the sample
scaler_f  = lr_tuned.named_steps['scaler']
X_te_sc   = scaler_f.transform(X_te)
X_shap_sc = pd.DataFrame(X_te_sc, columns=X_raw.columns).sample(1000, random_state=42)

explainer = shap.LinearExplainer(
    lr_tuned.named_steps['lr'],
    X_shap_sc,
    feature_perturbation='interventional'
)
shap_vals = explainer.shap_values(X_shap_sc)

feat_names = (
    pd.Index(X_raw.columns)
    .str.replace('Object of search_', '', regex=False)
    .str.replace('Age range_', 'Age: ', regex=False)
    .str.replace('Officer-defined ethnicity_', '', regex=False)
    .str.replace('Gender_', '', regex=False)
    .tolist()
)
print(f'SHAP values computed — shape: {shap_vals.shape}')

In [ ]:
# ── F2. Beeswarm plot ─────────────────────────────────────────────────────────
# Each dot = 1 prediction. x = SHAP value (push toward linked=1 or linked=0).
# Colour = feature value (red = high, blue = low).
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_vals, X_shap_sc.values,
                  feature_names=feat_names, max_display=15, show=False)
plt.title('SHAP Beeswarm — Binary LR\n'
          '(dot = 1 prediction  |  colour = feature value  |  x = impact on prediction)',
          fontsize=11)
plt.tight_layout()
plt.show()

# Summary: SHAP beeswarm plot reveals which features most influence the model’s predictions and how they push towards Linked (1) or Not Linked (0).

# Blue = feature absent e.g. not a drug search; red means: drug search. The further right the dot, the stronger the push towards Linked; the further left, the stronger the push towards Not Linked.
# Controlled drug searches is the strongest driver of Linked. High SHAP values (red) push predicitoin strongly towards Linked, while low SHAP values (blue) push towards Not Linked.
# Searches involving controeled drugs are highly predictive of linked cases, while searches without controlled drugs push towards not linked.
# Missing age (infomration) strong driver of Linked cases,
# Stolen good is a good predictor
# Age is another major driver    

In [ ]:
# ── F3. Global importance bar chart (mean |SHAP|) ─────────────────────────────
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_shap_sc.values,
                  feature_names=feat_names, plot_type='bar',
                  max_display=15, show=False)
plt.title('SHAP Global Importance — Binary LR  (mean |SHAP value|)', fontsize=12)
plt.tight_layout()
plt.show()

#Summary: Controlled drugs, followed by age: not recorded are the strongest predictors of Linked cases

---
## 11. Summary & Conclusions


### Conclusion Summary to date
#### Controlled drugs is the dominant search overall -31.42% happen betwen 3pm-6pm


In [ ]:
# Save cleaned dataset to CSV
output_path = 'stop_and_search_final_BTP.csv'
df_clean.to_csv(output_path, index=False)
print(f'Saved {len(df_clean):,} rows to {output_path}')